# M5/T2 Homework: Полноценная мультиагентная система с инструментами

В ноутбуке реализована мультиагентная система с ролями `planner`,
`researcher`, `coder`, `tester`, `reviewer`.
Инструменты работают в реальной файловой системе, а сценарий создаёт
артефакты в `M5/T2/artifacts`.

## 1. Настройка OpenRouter

In [1]:
import json
import os
import shutil
import subprocess
import sys
import textwrap
from collections import deque
from pathlib import Path
from typing import Any, Literal, Optional

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists():
            return candidate
    raise FileNotFoundError("Repository root with README.md was not found.")


REPO_ROOT = find_repo_root()
ARTIFACTS_DIR = REPO_ROOT / "M5" / "T2" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(REPO_ROOT / ".env", override=True)
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY was not found in the repository root .env file or environment.")

DEFAULT_MODEL = "openai/gpt-4o-mini"
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print("Repo root:", REPO_ROOT)
print("Artifacts dir:", ARTIFACTS_DIR)
print("Model:", DEFAULT_MODEL)

Repo root: /home/krv/repo/workspace/LLM_course
Artifacts dir: /home/krv/repo/workspace/LLM_course/M5/T2/artifacts
Model: openai/gpt-4o-mini


## 2. Вызов LLM с JSON Schema

Вызов модели сопровождается локальной Pydantic-валидацией ответа.

In [ ]:
def call_llm_with_schema(
    prompt: str,
    response_schema: type[BaseModel],
    system_prompt: Optional[str] = None,
    model: str = DEFAULT_MODEL,
) -> BaseModel:
    if openrouter_client is None:
        raise RuntimeError("OpenRouter client is not initialized.")

    schema_dict = response_schema.model_json_schema()
    schema_text = json.dumps(schema_dict, ensure_ascii=False, indent=2)
    current_prompt = prompt

    for attempt in range(3):
        messages: list[dict[str, str]] = []
        if system_prompt:
            messages.append(
                {
                    "role": "system",
                    "content": (
                        system_prompt
                        + "\n\nReturn ONLY valid JSON matching this schema:\n"
                        + schema_text
                    ),
                }
            )
        messages.append({"role": "user", "content": current_prompt})

        resp = openrouter_client.chat.completions.create(
            model=model,
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.1,
        )
        raw = (resp.choices[0].message.content or "").strip()
        if raw.startswith("```json"):
            raw = raw.split("```json", 1)[1].rsplit("```", 1)[0].strip()
        elif raw.startswith("```"):
            raw = raw.split("```", 1)[1].rsplit("```", 1)[0].strip()

        try:
            return response_schema.model_validate_json(raw)
        except (ValidationError, ValueError) as exc:
            if attempt == 2:
                raise
            current_prompt = (
                prompt
                + "\n\nYour previous answer failed schema validation:\n"
                + str(exc)
                + "\n\nReturn corrected JSON only."
            )

    raise RuntimeError("Unreachable retry branch.")


class DemoSchema(BaseModel):
    message: str


call_llm_with_schema

<function __main__.call_llm_with_schema(prompt: str, response_schema: type[pydantic.main.BaseModel], system_prompt: Optional[str] = None, model: str = 'openai/gpt-4o-mini') -> pydantic.main.BaseModel>

## 3. SGR: действия агента и сообщения

Протокол действий агента включает `AskAgent`, `UseTool`, `Reply`, `Finish`.

In [3]:
RoleName = Literal["planner", "researcher", "coder", "tester", "reviewer"]


class AskAgent(BaseModel):
    action: Literal["ask_agent"] = "ask_agent"
    target: RoleName
    message: str = Field(min_length=10)


class UseTool(BaseModel):
    action: Literal["use_tool"] = "use_tool"
    tool_name: str
    args: dict[str, Any] = Field(default_factory=dict)


class Reply(BaseModel):
    action: Literal["reply"] = "reply"
    content: str = Field(min_length=5)


class Finish(BaseModel):
    action: Literal["finish"] = "finish"
    summary: str = Field(min_length=10)


class AgentAction(BaseModel):
    step: AskAgent | UseTool | Reply | Finish


class BusMessage(BaseModel):
    sender: str
    recipient: str
    content: str


schema_demo = AgentAction(step={"action": "reply", "content": "Schema validation is ready."})
schema_demo

AgentAction(step=Reply(action='reply', content='Schema validation is ready.'))

## 4. Workspace и инструменты

Действия инструментов выполняются в реальной директории `M5/T2/artifacts`.

In [4]:
class Workspace:
    def __init__(self, root: Path):
        self.root = root
        self.root.mkdir(parents=True, exist_ok=True)
        self.memory: dict[str, Any] = {}
        self.last_test_result: dict[str, Any] = {}

    def reset(self) -> None:
        for path in self.root.glob("*"):
            if path.is_file():
                path.unlink()
            elif path.is_dir():
                shutil.rmtree(path)
        self.last_test_result = {}

    def normalize_path(self, filename: str) -> str:
        cleaned = str(filename or "").strip().replace("\\", "/")
        while cleaned.startswith("./"):
            cleaned = cleaned[2:]
        if cleaned.startswith("artifacts/"):
            cleaned = cleaned[len("artifacts/") :]
        return cleaned

    def resolve(self, filename: str) -> Path:
        cleaned = self.normalize_path(filename)
        if not cleaned:
            raise ValueError("filename must point to a file inside artifacts.")
        target = (self.root / cleaned).resolve()
        if self.root.resolve() not in [target, *target.parents]:
            raise ValueError(f"path escapes artifacts root: {filename}")
        if target.exists() and target.is_dir():
            raise ValueError(f"path points to a directory, not a file: {filename}")
        return target

    def list_files(self) -> dict[str, Any]:
        files = sorted(
            path.relative_to(self.root).as_posix()
            for path in self.root.rglob("*")
            if path.is_file() and "__pycache__" not in path.parts and path.suffix != ".pyc"
        )
        return {"files": files, "count": len(files)}

    def store_code(self, filename: str, code: str) -> dict[str, Any]:
        try:
            target = self.resolve(filename)
        except ValueError as exc:
            return {"error": str(exc)}
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(code, encoding="utf-8")
        return {"filename": self.normalize_path(filename), "chars": len(code), "preview": code[:300]}

    def write_report(self, filename: str, content: str) -> dict[str, Any]:
        try:
            target = self.resolve(filename)
        except ValueError as exc:
            return {"error": str(exc)}
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(content, encoding="utf-8")
        return {"filename": self.normalize_path(filename), "chars": len(content), "preview": content[:300]}

    def read_code(self, filename: str) -> dict[str, Any]:
        try:
            target = self.resolve(filename)
        except ValueError as exc:
            return {"error": str(exc)}
        if not target.exists():
            return {"error": f"file not found: {self.normalize_path(filename)}"}
        return {"filename": self.normalize_path(filename), "code": target.read_text(encoding="utf-8")}

    def run_python(self, filename: str) -> dict[str, Any]:
        try:
            target = self.resolve(filename)
        except ValueError as exc:
            return {"error": str(exc)}
        if not target.exists():
            return {"error": f"file not found: {self.normalize_path(filename)}"}

        env = os.environ.copy()
        current_pythonpath = env.get("PYTHONPATH", "")
        env["PYTHONPATH"] = str(self.root) + (os.pathsep + current_pythonpath if current_pythonpath else "")

        proc = subprocess.run(
            [sys.executable, str(target)],
            cwd=self.root,
            capture_output=True,
            text=True,
            timeout=30,
            env=env,
        )
        return {
            "filename": self.normalize_path(filename),
            "returncode": proc.returncode,
            "stdout": proc.stdout[-4000:],
            "stderr": proc.stderr[-4000:],
        }

    def run_tests(self, filename: str) -> dict[str, Any]:
        payload = self.run_python(filename)
        payload["passed"] = payload.get("returncode") == 0
        self.last_test_result = payload
        return payload

    def write_json(self, filename: str, data: dict[str, Any]) -> dict[str, Any]:
        try:
            target = self.resolve(filename)
        except ValueError as exc:
            return {"error": str(exc)}
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")
        return {"filename": self.normalize_path(filename), "keys": sorted(data.keys())}


class Tools:
    def __init__(self, ws: Workspace):
        self.ws = ws

    def call(self, tool_name: str, args: dict[str, Any]) -> dict[str, Any]:
        args = args or {}
        if tool_name == "list_files":
            return self.ws.list_files()
        if tool_name == "store_code":
            return self.ws.store_code(filename=args.get("filename", ""), code=args.get("code", ""))
        if tool_name == "read_code":
            return self.ws.read_code(filename=args.get("filename", ""))
        if tool_name == "run_python":
            return self.ws.run_python(filename=args.get("filename", ""))
        if tool_name == "run_tests":
            return self.ws.run_tests(filename=args.get("filename", ""))
        if tool_name == "write_report":
            return self.ws.write_report(filename=args.get("filename", ""), content=args.get("content", ""))
        if tool_name == "write_json":
            return self.ws.write_json(filename=args.get("filename", ""), data=args.get("data", {}))
        return {"error": f"unknown tool {tool_name}"}


ws = Workspace(ARTIFACTS_DIR)
tools = Tools(ws)


def reset_artifacts() -> None:
    ws.reset()
    print("Artifacts directory cleared.")
    print(tools.call("list_files", {}))


print(tools.call("list_files", {}))

{'files': ['number_stats.py', 'report.md', 'run_summary.json', 'test_number_stats.py'], 'count': 4}


## 5. Локальная проверка инструментов

Небольшой smoke test до запуска мультиагентной части.

In [5]:
tool_smoke = {}
tool_smoke["store_code"] = tools.call(
    "store_code",
    {"filename": "smoke.py", "code": "print('smoke')\n"},
)
tool_smoke["read_code"] = tools.call("read_code", {"filename": "smoke.py"})
tool_smoke["write_json"] = tools.call(
    "write_json",
    {"filename": "smoke.json", "data": {"status": "ok"}},
)
tool_smoke["list_files"] = tools.call("list_files", {})
tool_smoke

{'store_code': {'filename': 'smoke.py',
  'chars': 15,
  'preview': "print('smoke')\n"},
 'read_code': {'filename': 'smoke.py', 'code': "print('smoke')\n"},
 'write_json': {'filename': 'smoke.json', 'keys': ['status']},
 'list_files': {'files': ['number_stats.py',
   'report.md',
   'run_summary.json',
   'smoke.json',
   'smoke.py',
   'test_number_stats.py'],
  'count': 6}}

## 6. Шина сообщений и агенты

У каждого агента есть роль, ограниченный набор инструментов и метод
`decide(...)`, который возвращает `AgentAction`.

In [6]:
class MessageBus:
    def __init__(self):
        self.queue: deque[BusMessage] = deque()
        self.history: list[BusMessage] = []

    def send(self, msg: BusMessage):
        self.queue.append(msg)
        self.history.append(msg)

    def receive_for(self, recipient: str) -> list[BusMessage]:
        msgs = [m for m in list(self.queue) if m.recipient in (recipient, "broadcast")]
        for m in msgs:
            try:
                self.queue.remove(m)
            except ValueError:
                pass
        return msgs


bus = MessageBus()


ROLE_RULES: dict[str, str] = {
    "planner": textwrap.dedent(
        """
        Вы — планировщик. Работайте строго по этапам:
        1. попросите researcher подготовить research_notes.json;
        2. попросите coder создать number_stats.py;
        3. попросите tester создать test_number_stats.py и запустить run_tests;
        4. после успешных тестов попросите reviewer создать report.md и run_summary.json;
        5. завершите процесс только когда есть все обязательные файлы и тесты прошли.

        Обычно выбирайте ask_agent. Используйте finish только в самом конце.
        """
    ).strip(),
    "researcher": textwrap.dedent(
        """
        Вы — аналитик требований. Не пишите код модулей и тестов.
        Когда planner просит исследование:
        - сначала используйте tool write_json, чтобы сохранить research_notes.json;
        - затем сделайте reply с кратким summary для planner.

        В research_notes.json сохраните ключи:
        - requirements
        - edge_cases
        - acceptance_criteria
        """
    ).strip(),
    "coder": textwrap.dedent(
        """
        Вы — разработчик.
        Когда planner или tester просит реализацию или исправление:
        - при необходимости сначала прочитайте текущий number_stats.py через read_code;
        - затем используйте store_code и создайте number_stats.py.

        Требования к коду:
        - функции mean_value, median_value, range_width, describe_numbers;
        - пустой список должен вызывать ValueError;
        - describe_numbers возвращает dict с ключами count, mean, median, min, max, range_width.
        """
    ).strip(),
    "tester": textwrap.dedent(
        """
        Вы — тестировщик.
        Когда код готов:
        - сначала прочитайте number_stats.py;
        - затем через store_code создайте test_number_stats.py;
        - затем через run_tests запустите test_number_stats.py.

        Тесты должны:
        - использовать plain assert;
        - проверять нормальный кейс и empty-list failure case;
        - печатать ровно TESTS_PASSED при успехе.

        Если тесты не проходят, отправьте ask_agent агенту coder с кратким описанием ошибки.
        Если тесты проходят, сделайте reply.
        """
    ).strip(),
    "reviewer": textwrap.dedent(
        """
        Вы — reviewer.
        Когда тесты прошли:
        - прочитайте number_stats.py и test_number_stats.py;
        - используйте write_report для report.md;
        - используйте write_json для run_summary.json;
        - затем сделайте reply.

        report.md должен кратко описывать:
        - архитектуру системы;
        - созданные файлы;
        - результат тестирования.
        """
    ).strip(),
}


class LlmAgent:
    def __init__(self, name: str, system_role: str, allowed_tools: list[str]):
        self.name = name
        self.system_role = system_role
        self.allowed_tools = allowed_tools

    def decide(self, goal: str, inbox: list[BusMessage], state: dict[str, Any]) -> AgentAction:
        history_text = "\n".join(f"{m.sender}→{m.recipient}: {m.content}" for m in bus.history[-12:])
        inbox_text = "\n".join(f"{m.sender}→{m.recipient}: {m.content}" for m in inbox[-6:]) or "(пусто)"
        tools_str = ", ".join(self.allowed_tools) if self.allowed_tools else "(нет)"
        prompt = f"""
Вы — {self.system_role}. Ваша роль: {self.name}.

Цель:
{goal}

Текущее состояние:
{json.dumps(state, ensure_ascii=False, indent=2)}

Ваш inbox:
{inbox_text}

История:
{history_text}

Доступные инструменты:
{tools_str}

Выберите ровно одно действие:
- ask_agent (target, message)
- use_tool (tool_name in [{tools_str}], args)
- reply (content)
- finish (summary)

Правила:
- не придумывайте недоступные инструменты;
- если задача вашей роли требует создания файла, сначала делайте use_tool;
- finish разрешён только planner и только когда задача действительно завершена;
- отвечайте только JSON по схеме AgentAction.
"""
        return call_llm_with_schema(prompt=prompt, response_schema=AgentAction, system_prompt=self.system_role)

## 7. Оркестратор

Оркестратор централизованный:
- роли и `allowed_tools` задаются явно;
- каждый агент делает ровно один шаг за ход;
- проверка доступа к инструментам идёт на уровне оркестратора;
- весь прогон логируется в `bus.history`.

In [ ]:
class Orchestrator:
    def __init__(self):
        self.agents = {
            "planner": LlmAgent("planner", ROLE_RULES["planner"], []),
            "researcher": LlmAgent("researcher", ROLE_RULES["researcher"], ["write_json", "list_files"]),
            "coder": LlmAgent("coder", ROLE_RULES["coder"], ["store_code", "read_code", "list_files"]),
            "tester": LlmAgent("tester", ROLE_RULES["tester"], ["store_code", "read_code", "run_tests", "list_files"]),
            "reviewer": LlmAgent("reviewer", ROLE_RULES["reviewer"], ["read_code", "write_report", "write_json", "list_files"]),
        }
        self.allowed_tools = {name: agent.allowed_tools for name, agent in self.agents.items()}

    def state_snapshot(self) -> dict[str, Any]:
        files = ws.list_files().get("files", [])
        return {
            "files": files,
            "research_ready": "research_notes.json" in files,
            "code_ready": "number_stats.py" in files,
            "tests_written": "test_number_stats.py" in files,
            "report_ready": "report.md" in files,
            "summary_ready": "run_summary.json" in files,
            "tests_passed": bool(ws.last_test_result.get("passed")),
            "last_test_result": ws.last_test_result,
        }

    def completion_ready(self) -> bool:
        state = self.state_snapshot()
        required = {"number_stats.py", "test_number_stats.py", "report.md", "run_summary.json"}
        return required.issubset(set(state["files"])) and state["tests_passed"]

    def step(self, goal: str, order: list[str]) -> bool:
        progressed = False
        for name in order:
            inbox = bus.receive_for(name)
            if name != "planner" and not inbox:
                continue

            if inbox:
                print(f"[{name}] inbox ({len(inbox)}):")
                for m in inbox[-3:]:
                    print(f"  {m.sender}→{m.recipient}: {m.content[:220]}")

            state = self.state_snapshot()
            action = self.agents[name].decide(goal=goal, inbox=inbox, state=state)
            s = action.step
            print(f"[{name}] action: {s}")

            if isinstance(s, AskAgent):
                bus.send(BusMessage(sender=name, recipient=s.target, content=s.message))
                print(f"[{name}] → [{s.target}] ask: {s.message}")
                progressed = True
                continue

            if isinstance(s, UseTool):
                if s.tool_name not in self.allowed_tools.get(name, []):
                    msg = (
                        f"роль {name} не имеет доступа к инструменту {s.tool_name}. "
                        f"Доступны: {self.allowed_tools.get(name, [])}"
                    )
                    print(f"[{name}] denied {s.tool_name}: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))
                    continue

                result = tools.call(s.tool_name, s.args)
                print(f"[{name}] used {s.tool_name} → {str(result)[:400]}")
                bus.send(
                    BusMessage(
                        sender=name,
                        recipient="broadcast",
                        content=f"tool {s.tool_name} result: {json.dumps(result, ensure_ascii=False)}",
                    )
                )
                progressed = True
                continue

            if isinstance(s, Reply):
                bus.send(BusMessage(sender=name, recipient="broadcast", content=s.content))
                print(f"[{name}] reply: {s.content}")
                progressed = True
                continue

            if isinstance(s, Finish):
                if name != "planner":
                    msg = "finish разрешён только planner"
                    print(f"[{name}] denied finish: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient=name, content=msg))
                    continue
                if not self.completion_ready():
                    msg = "finish denied: обязательные артефакты ещё не готовы или тесты не прошли"
                    print(f"[planner] denied finish: {msg}")
                    bus.send(BusMessage(sender="orchestrator", recipient="planner", content=msg))
                    continue
                bus.send(BusMessage(sender=name, recipient="broadcast", content=f"FINISH: {s.summary}"))
                print(f"[{name}] finish: {s.summary}")
                return True

        if not progressed:
            bus.send(
                BusMessage(
                    sender="orchestrator",
                    recipient="planner",
                    content="Нет прогресса в текущем раунде. Продвиньте следующий нужный этап.",
                )
            )
        return False

    def run(self, goal: str, max_rounds: int = 12, reset_workspace: bool = True) -> dict[str, Any]:
        print("=== ORCHESTRATION START ===")
        if reset_workspace:
            ws.reset()
        bus.queue.clear()
        bus.history.clear()
        ws.last_test_result = {}

        bus.send(BusMessage(sender="planner", recipient="broadcast", content=f"start: {goal}"))
        for round_idx in range(1, max_rounds + 1):
            print(f"\n--- ROUND {round_idx} ---")
            if self.step(goal, ["planner", "researcher", "coder", "tester", "reviewer"]):
                print("=== ORCHESTRATION FINISHED ===")
                break
        else:
            print("=== MAX ROUNDS REACHED ===")

        return {
            "finished": self.completion_ready(),
            "files": ws.list_files().get("files", []),
            "last_test_result": ws.last_test_result,
            "history_length": len(bus.history),
        }


orch = Orchestrator()

## 8. Целевой сценарий

Сценарий: Planner → Researcher → Coder → Tester → Reviewer.

In [8]:
goal = textwrap.dedent(
    """
    Построй мультиагентную систему, которая создаст и проверит маленький Python-проект в artifacts.

    Обязательные deliverables:
    1. number_stats.py
       - Реализовать mean_value(values), median_value(values), range_width(values), describe_numbers(values).
       - Для пустого списка выбрасывать ValueError.
       - describe_numbers должен возвращать dict с ключами count, mean, median, min, max, range_width.

    2. test_number_stats.py
       - Использовать plain assert.
       - Проверить нормальный кейс и пустой список.
       - При успехе печатать ровно TESTS_PASSED.

    3. report.md
       - Кратко описать архитектуру системы, созданные файлы и результат тестов.

    4. run_summary.json
       - Сохранить структурированный итог прогона.

    Дополнительно:
    - researcher должен подготовить research_notes.json;
    - все пути должны быть относительными к artifacts;
    - завершение разрешено только после успешных тестов и записи итоговых артефактов.
    """
).strip()

print(goal)

Построй мультиагентную систему, которая создаст и проверит маленький Python-проект в artifacts.

Обязательные deliverables:
1. number_stats.py
   - Реализовать mean_value(values), median_value(values), range_width(values), describe_numbers(values).
   - Для пустого списка выбрасывать ValueError.
   - describe_numbers должен возвращать dict с ключами count, mean, median, min, max, range_width.

2. test_number_stats.py
   - Использовать plain assert.
   - Проверить нормальный кейс и пустой список.
   - При успехе печатать ровно TESTS_PASSED.

3. report.md
   - Кратко описать архитектуру системы, созданные файлы и результат тестов.

4. run_summary.json
   - Сохранить структурированный итог прогона.

Дополнительно:
- researcher должен подготовить research_notes.json;
- все пути должны быть относительными к artifacts;
- завершение разрешено только после успешных тестов и записи итоговых артефактов.


In [9]:
run_result = orch.run(goal=goal, max_rounds=12, reset_workspace=True)
run_result

=== ORCHESTRATION START ===

--- ROUND 1 ---
[planner] inbox (1):
  planner→broadcast: start: Построй мультиагентную систему, которая создаст и проверит маленький Python-проект в artifacts.

Обязательные deliverables:
1. number_stats.py
   - Реализовать mean_value(values), median_value(values), range_width
[planner] action: action='ask_agent' target='researcher' message='Пожалуйста, подготовьте файл research_notes.json.'
[planner] → [researcher] ask: Пожалуйста, подготовьте файл research_notes.json.
[researcher] inbox (1):
  planner→researcher: Пожалуйста, подготовьте файл research_notes.json.
[researcher] action: action='use_tool' tool_name='write_json' args={'requirements': {'number_stats.py': {'functions': ['mean_value(values)', 'median_value(values)', 'range_width(values)', 'describe_numbers(values)'], 'error_handling': 'ValueError для пустого списка', 'describe_numbers_return': {'type': 'dict', 'keys': ['count', 'mean', 'median', 'min', 'max', 'range_width']}}, 'test_number_stats.

{'finished': True,
 'files': ['number_stats.py',
  'report.md',
  'research_notes.json',
  'run_summary.json',
  'test_number_stats.py'],
 'last_test_result': {'filename': 'test_number_stats.py',
  'returncode': 0,
  'stdout': 'TESTS_PASSED\n',
  'stderr': '',
  'passed': True},
 'history_length': 16}

In [10]:
required_files = {"research_notes.json", "number_stats.py", "test_number_stats.py", "report.md", "run_summary.json"}
created_files = set(run_result["files"])
missing_files = sorted(required_files - created_files)

print("Missing files:", missing_files)
print("Tests passed:", run_result["last_test_result"].get("passed"))

assert not missing_files, f"Missing required files: {missing_files}"
assert run_result["last_test_result"].get("passed") is True, "Tests did not pass."
assert run_result["finished"] is True, "The orchestration did not reach a valid finish state."

Missing files: []
Tests passed: True


## 9. Логи и артефакты

Здесь видны сообщения между агентами и итоговые файлы.

In [11]:
print("Bus history:")
for m in bus.history:
    print(f"- {m.sender}→{m.recipient}: {m.content[:260]}")

Bus history:
- planner→broadcast: start: Построй мультиагентную систему, которая создаст и проверит маленький Python-проект в artifacts.

Обязательные deliverables:
1. number_stats.py
   - Реализовать mean_value(values), median_value(values), range_width(values), describe_numbers(values).
   -
- planner→researcher: Пожалуйста, подготовьте файл research_notes.json.
- researcher→broadcast: tool write_json result: {"error": "filename must point to a file inside artifacts."}
- coder→broadcast: tool store_code result: {"filename": "number_stats.py", "chars": 849, "preview": "def mean_value(values):\n    if not values:\n        raise ValueError('List is empty')\n    return sum(values) / len(values)\n\ndef median_value(values):\n    if not values:\n   
- tester→broadcast: tool store_code result: {"filename": "test_number_stats.py", "chars": 1636, "preview": "import number_stats\n\ndef test_mean_value():\n    assert number_stats.mean_value([1, 2, 3]) == 2.0\n    assert number_stats.mean_value(

In [12]:
artifact_listing = tools.call("list_files", {})
artifact_listing

{'files': ['number_stats.py',
  'report.md',
  'research_notes.json',
  'run_summary.json',
  'test_number_stats.py'],
 'count': 5}

In [ ]:
for filename in artifact_listing.get("files", []):
    print("=" * 80)
    print(filename)
    print("=" * 80)
    print(tools.call("read_code", {"filename": filename}).get("code", "")[:4000])

## 10. Краткий отчёт

**Что реализовано**
- Архитектура включает `Workspace`, `Tools`, `MessageBus`, `LlmAgent`, `Orchestrator`.
- В качестве SGR-протокола используются валидируемые схемы `AskAgent`, `UseTool`, `Reply`, `Finish`.
- Роли разделены по обязанностям и инструментам:
  - `planner` координирует шаги;
  - `researcher` отвечает за `research_notes.json`;
  - `coder` создаёт `number_stats.py`;
  - `tester` создаёт `test_number_stats.py` и запускает тесты;
  - `reviewer` создаёт `report.md` и `run_summary.json`.

**Что показал фактический прогон**
- В конце прогона были созданы `research_notes.json`, `number_stats.py`, `test_number_stats.py`, `report.md`, `run_summary.json`.
- Тесты завершились успешно: `run_result["last_test_result"]["passed"] == True`, в stdout получено `TESTS_PASSED`.
- В логах видны адресные сообщения между ролями, вызовы инструментов и финальное сообщение `FINISH`.

**Наблюдения по ходу выполнения**
- Прогон не был полностью линейным: сначала `researcher` не сохранил `research_notes.json`, затем `reviewer` не сохранил `report.md`, а planner попытался завершить процесс раньше времени.
- Эти шаги были скорректированы в следующих раундах, после чего система дошла до корректного завершения.
- Таким образом, сценарий демонстрирует не только успешный finish, но и обработку промежуточных неудачных шагов через повторные ходы и проверки оркестратора.

**Проверка артефактов**
- Просмотрены все итоговые файлы: `research_notes.json`, `number_stats.py`, `test_number_stats.py`, `report.md`, `run_summary.json`.
- `research_notes.json` содержит требования, edge cases и критерии приёмки для реализации.
- `number_stats.py` реализует требуемые функции для вычисления статистик и корректно обрабатывает пустой список через `ValueError`.
- `test_number_stats.py` содержит проверки основных сценариев и завершается сообщением `TESTS_PASSED`.
- `report.md` фиксирует архитектуру решения, а `run_summary.json` сохраняет структурированный результат тестового прогона.